In [0]:
from datetime import datetime
from pyspark.sql.functions import current_timestamp

In [0]:
df = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','parquet')\
    .option("cloudFiles.inferColumnTypes", "true")\
    .option("cloudFiles.schemaEvolutionMode", "rescue")\
    .option('cloudFiles.schemaLocation','/Volumes/projectdev/bronze/cancellation/schema')\
    .load(f'abfss://raw@datalakestrgdev.dfs.core.windows.net/Cancellation/{datetime.now().strftime('year=%Y/month=%m/day=%d')}/')

In [0]:
from pyspark.sql.functions import regexp_replace, current_timestamp, col

df_clean = (
    df.withColumn("Code", regexp_replace(col("Code"), '"', ""))
      .withColumn("Description", regexp_replace(col("Description"), '"', ""))
      .withColumn("last_load_ts", current_timestamp())
)

from delta.tables import DeltaTable

target_table = 'projectdev.cleansed.cancellation'
def upsert_to_delta(microBatchDF, batchId):
    delta_table = DeltaTable.forName(spark,target_table)
    if not spark.catalog.tableExists(target_table):
        microBatchDF.write.mode("overwrite").format("delta").saveAsTable(target_table)
    else:
        (
            delta_table.alias("t")
            .merge(
                microBatchDF.alias("s"),
                "t.Code = s.Code"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

query = (
    df_clean.writeStream
        .trigger(once=True)
        .option('mergeSchema','true')
        .foreachBatch(upsert_to_delta)
        .option("checkpointLocation", '/Volumes/projectdev/bronze/cancellation/checkpoint/')
        .start()
)

query.awaitTermination()

In [0]:
%sql
select * from projectdev.cleansed.cancellation